# **Limpieza de Datos con Pandas**

La limpieza de datos es una de las etapas más críticas en cualquier proyecto de análisis o ciencia de datos. Un dataset sucio puede arruinar el modelo más sofisticado.

En este cuaderno trabajaremos con un dataset de ventas, cubriendo las siguientes tareas:

1. Exploración inicial
2. Limpieza de cadenas de texto
3. Identificación y manejo de valores faltantes
4. Corrección de tipos de datos (incluyendo fechas)
5. Identificación de Valores Sospechosos (outliers)
6. Eliminación de duplicados
7. Validación final del dataset limpio
8. Exportación a csv


## Dataset: Ventas de Productos

El dataset simula registros de ventas con problemas comunes del mundo real: valores nulos, errores de tipeo, formatos inconsistentes y duplicados.


In [1]:
import pandas as pd
import numpy as np

data = {
    "Id_venta":           [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1010, 1011, 1012],
    "Id_producto":        ["P-01", "P-02", "P-03", "P-01", "P-04", "P-02", "P-03", "p-04", "P-05", "P-01", "P-01", "P-06", None],
    "Categoria":          ["Electrónica", "Ropa", "Ropa", "electrónica", None, "ROPA", "Ropa", "Electrónica", "Hogar", "Electrónica", "Electrónica", None, None],
    "CantidadVendida":   [30, None, 25, 10, 8, None, 15, 5, 12, 9, 9, 7, None],
    "PrecioUnitario":    [20.5, 15.0, None, 20.5, 999.0, 15.0, 22.5, 18.0, None, 20.5, 20.5, 35.0, None],
    "Vendedor":           ["  Ana García", "Luis Pérez", "Ana García  ", "Luis Pérez", "  Carlos López", "Carlos López", "Ana García", "LUIS PÉREZ", "Ana García", "Luis Pérez", "Luis Pérez", None, "  "],
    "Region":             ["Norte", "Sur", "Norte", "Norte", "Centro", "Sur", None, "Centro", "Norte", "Norte", "Norte", "Sur", None],
    "Fecha_venta":        ["2024-01-15", "2024-01-16", "15/01/2024", "2024-01-18", "2024-01-19", "2024-01-20", "2024-01-21", "2024-01-22", "2024-01-23", "2024-01-24", "2024-01-24", "2024-01-25", None],
    "Notas_devolucion":   [None, None, "Talle incorrecto", None, None, None, None, "Producto defectuoso", None, None, None, None, None],
}

df = pd.DataFrame(data)
df


,Id_venta,Id_producto,Categoria,CantidadVendida,PrecioUnitario,Vendedor,Region,Fecha_venta,Notas_devolucion
0,1001,P-01,Electrónica,30.0,20.5,Ana García,Norte,2024-01-15,None
1,1002,P-02,Ropa,NaN,15.0,Luis Pérez,Sur,2024-01-16,None
2,1003,P-03,Ropa,25.0,NaN,Ana García,Norte,15/01/2024,Talle incorrecto
3,1004,P-01,electrónica,10.0,20.5,Luis Pérez,Norte,2024-01-18,None
4,1005,P-04,None,8.0,999.0,Carlos López,Centro,2024-01-19,None
5,1006,P-02,ROPA,NaN,15.0,Carlos López,Sur,2024-01-20,None
6,1007,P-03,Ropa,15.0,22.5,Ana García,None,2024-01-21,None
7,1008,p-04,Electrónica,5.0,18.0,LUIS PÉREZ,Centro,2024-01-22,Producto defectuoso
8,1009,P-05,Hogar,12.0,NaN,Ana García,Norte,2024-01-23,None
9,1010,P-01,Electrónica,9.0,20.5,Luis Pérez,Norte,2024-01-24,None


---
## Paso 1: Exploración Inicial

Antes de limpiar, necesitamos entender qué tenemos. El objetivo es responder:
- ¿Cuántas filas y columnas tiene el dataset?
- ¿Qué tipos de datos hay?
- ¿Cuáles son los rangos y distribuciones básicas?


In [18]:
# Dimensiones del dataset
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")


Filas: 13 | Columnas: 9


In [19]:
# Tipos de datos y valores no nulos por columna
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id_venta          13 non-null     int64  
 1   Id_producto       12 non-null     object 
 2   Categoria         10 non-null     object 
 3   CantidadVendida   10 non-null     float64
 4   PrecioUnitario    10 non-null     float64
 5   Vendedor          12 non-null     object 
 6   Region            11 non-null     object 
 7   Fecha_venta       12 non-null     object 
 8   Notas_devolucion  2 non-null      object 
dtypes: float64(2), int64(1), object(6)
memory usage: 1.0+ KB


In [20]:
# Estadísticas descriptivas de columnas numéricas
df.describe()


,Id_venta,CantidadVendida,PrecioUnitario
count,13.000000,10.000000,10.000000
mean,1006.769231,13.000000,118.650000
std,3.585941,8.192137,309.373745
min,1001.000000,5.000000,15.000000
25%,1004.000000,8.250000,18.625000
50%,1007.000000,9.500000,20.500000
75%,1010.000000,14.250000,22.000000
max,1012.000000,30.000000,999.000000


In [5]:
# Frecuencia de categorías
print("Categorías únicas encontradas:")
print(df["Categoria"].value_counts(dropna=False))


Categorías únicas encontradas:
Categoria
Electrónica    4
Ropa           3
None           3
electrónica    1
ROPA           1
Hogar          1
Name: count, dtype: int64


In [21]:
# Frecuencia de vendedores
print("Vendedores:")
print(df["Vendedor"].value_counts(dropna=False))

Vendedores:
Vendedor
Luis Pérez        4
Ana García        2
  Ana García      1
  Carlos López    1
Ana García        1
Carlos López      1
LUIS PÉREZ        1
None              1
                  1
Name: count, dtype: int64


## Regla: Primero limpio, después analizo los nulos y finalmente los trato (eliminando o imputando)

---
## Paso 2: Limpieza de Cadenas de Texto

Los errores de tipeo y formato inconsistente en columnas de texto son extremadamente comunes. Hay que normalizarlos antes de cualquier análisis o agrupación.


In [22]:
# Normalizar Categoria: eliminar espacios y capitalizar
df_limpio = df.copy()
df_limpio["Categoria"] = df_limpio["Categoria"].str.strip().str.capitalize()
print("Categorías únicas después de limpiar:")
print(df_limpio["Categoria"].value_counts())


Categorías únicas después de limpiar:
Categoria
Electrónica    5
Ropa           4
Hogar          1
Name: count, dtype: int64


In [8]:
df_limpio["Id_producto"]

,Id_producto
0,P-01
1,P-02
2,P-03
3,P-01
4,P-04
5,P-02
6,P-03
7,p-04
8,P-05
9,P-01


In [23]:
# Normalizar Id_producto: todo en mayúsculas y sin espacios
df_limpio["Id_producto"] = df_limpio["Id_producto"].str.strip().str.upper()

In [24]:
# Normalizar Vendedor: eliminar espacios y usar formato título
df_limpio["Vendedor"] = df_limpio["Vendedor"].str.strip().str.title()

print("Vendedores únicos después de limpiar:")
print(df_limpio["Vendedor"].value_counts())

Vendedores únicos después de limpiar:
Vendedor
Luis Pérez      5
Ana García      4
Carlos López    2
                1
Name: count, dtype: int64


In [25]:
df_limpio

,Id_venta,Id_producto,Categoria,CantidadVendida,PrecioUnitario,Vendedor,Region,Fecha_venta,Notas_devolucion
0,1001,P-01,Electrónica,30.0,20.5,Ana García,Norte,2024-01-15,None
1,1002,P-02,Ropa,NaN,15.0,Luis Pérez,Sur,2024-01-16,None
2,1003,P-03,Ropa,25.0,NaN,Ana García,Norte,15/01/2024,Talle incorrecto
3,1004,P-01,Electrónica,10.0,20.5,Luis Pérez,Norte,2024-01-18,None
4,1005,P-04,None,8.0,999.0,Carlos López,Centro,2024-01-19,None
5,1006,P-02,Ropa,NaN,15.0,Carlos López,Sur,2024-01-20,None
6,1007,P-03,Ropa,15.0,22.5,Ana García,None,2024-01-21,None
7,1008,P-04,Electrónica,5.0,18.0,Luis Pérez,Centro,2024-01-22,Producto defectuoso
8,1009,P-05,Hogar,12.0,NaN,Ana García,Norte,2024-01-23,None
9,1010,P-01,Electrónica,9.0,20.5,Luis Pérez,Norte,2024-01-24,None


---
## Paso 3: Identificación y Manejo de Valores Faltantes

### 3.1 Cuantificar los nulos


In [26]:
# Conteo y porcentaje de nulos por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(1)

resumen_nulos = pd.DataFrame({
    "Nulos": nulos,
    "Porcentaje (%)": porcentaje
}).query("Nulos > 0")

resumen_nulos


,Nulos,Porcentaje (%)
Id_producto,1,7.7
Categoria,3,23.1
CantidadVendida,3,23.1
PrecioUnitario,3,23.1
Vendedor,1,7.7
Region,2,15.4
Fecha_venta,1,7.7
Notas_devolucion,11,84.6


### 3.2 Eliminación de filas demasiado incompletas

No siempre conviene imputar  (reemplazar usando media, mediana o moda según el tipo de variable).  
Cuando un registro tiene demasiados valores faltantes, suele ser más seguro eliminarlo.

**Regla práctica:**
- pocos nulos → imputar
- muchos nulos en columna → eliminar columna
- muchos nulos en fila → eliminar fila

La técnica de **imputación** depende de la naturaleza de los datos y de su distribución.

- **Media (promedio)**: se utiliza en variables numéricas cuando los datos están relativamente equilibrados y no presentan valores extremos que alteren significativamente el promedio.
- **Mediana (valor central)**: es más adecuada cuando existen valores atípicos, ya que no se ve afectada por extremos y representa mejor el centro real de los datos.
- **Moda (valor más frecuente)**: se aplica principalmente en variables categóricas, ya que permite reemplazar valores faltantes por la categoría con mayor presencia en el conjunto de datos.

In [27]:
# Eliminar filas con menos del 50% de datos presentes
# min_porcentaje_datos es el mínimo de valores NO nulos que debe tener la fila para conservarse
min_porcentaje_datos = 0.50
min_columnas_validas = int(min_porcentaje_datos * df_limpio.shape[1])

print(f"Filas con más del {int(min_porcentaje_datos*100)}% de nulos → se eliminan")
print("Filas antes:", len(df_limpio))

df_limpio = df_limpio.dropna(thresh=min_columnas_validas)

print("Filas después:", len(df_limpio))

Filas con más del 50% de nulos → se eliminan
Filas antes: 13
Filas después: 12


### 3.3 Eliminación de columnas demasiado incompletas


In [28]:
# Caso > 50% de nulos: eliminar la columna
porcentaje_max_nulos = 0.50
nulos_por_col = df_limpio.isna().mean()

nulos_por_col



,0
Id_venta,0.000000
Id_producto,0.000000
Categoria,0.166667
CantidadVendida,0.166667
PrecioUnitario,0.166667
Vendedor,0.083333
Region,0.083333
Fecha_venta,0.000000
Notas_devolucion,0.833333


In [29]:
cols_a_eliminar = nulos_por_col[nulos_por_col > porcentaje_max_nulos].index.tolist()
print(f"Columnas con más del {int(porcentaje_max_nulos*100)}% de nulos → se eliminan: {cols_a_eliminar}")

df_limpio = df_limpio.drop(columns=cols_a_eliminar)

print(f"Columnas restantes: {list(df_limpio.columns)}\n")

Columnas con más del 50% de nulos → se eliminan: ['Notas_devolucion']
Columnas restantes: ['Id_venta', 'Id_producto', 'Categoria', 'CantidadVendida', 'PrecioUnitario', 'Vendedor', 'Region', 'Fecha_venta']



### 3.4 Estrategia de imputación



In [31]:
# Caso < 50% de nulos: imputar

# Categoria (categórica) → imputar con moda
moda_categoria = df_limpio["Categoria"].mode()[0]
df_limpio["Categoria"] = df_limpio["Categoria"].fillna(moda_categoria)
print(f"Categoria imputada con moda: '{moda_categoria}'")

# Region (categórica) → imputar con moda
moda_region = df_limpio["Region"].mode()[0]
df_limpio["Region"] = df_limpio["Region"].fillna(moda_region)
print(f"Region imputada con moda: '{moda_region}'")

# Vendedor (categórica) → imputar con moda
moda_vendedor = df_limpio["Vendedor"].str.strip().mode()[0]
df_limpio["Vendedor"] = df_limpio["Vendedor"].fillna(moda_vendedor)
print(f"Vendedor imputado con moda: '{moda_vendedor}'")

# Variables numéricas → imputar con media
media_cantidad = df_limpio["CantidadVendida"].mean()
df_limpio["CantidadVendida"] = df_limpio["CantidadVendida"].fillna(media_cantidad)
print(f"Cantidad vendida imputada con media: {media_cantidad:.1f}")

media_precio = df_limpio["PrecioUnitario"].mean()
df_limpio["PrecioUnitario"] = df_limpio["PrecioUnitario"].fillna(media_precio)
print(f"Precio unitario imputado con media: {media_precio:.2f}")


Categoria imputada con moda: 'Electrónica'
Region imputada con moda: 'Norte'
Vendedor imputado con moda: 'Luis Pérez'
Cantidad vendida imputada con media: 13.0
Precio unitario imputado con media: 118.65


In [32]:
df_limpio

,Id_venta,Id_producto,Categoria,CantidadVendida,PrecioUnitario,Vendedor,Region,Fecha_venta
0,1001,P-01,Electrónica,30.0,20.50,Ana García,Norte,2024-01-15
1,1002,P-02,Ropa,13.0,15.00,Luis Pérez,Sur,2024-01-16
2,1003,P-03,Ropa,25.0,118.65,Ana García,Norte,15/01/2024
3,1004,P-01,Electrónica,10.0,20.50,Luis Pérez,Norte,2024-01-18
4,1005,P-04,Electrónica,8.0,999.00,Carlos López,Centro,2024-01-19
5,1006,P-02,Ropa,13.0,15.00,Carlos López,Sur,2024-01-20
6,1007,P-03,Ropa,15.0,22.50,Ana García,Norte,2024-01-21
7,1008,P-04,Electrónica,5.0,18.00,Luis Pérez,Centro,2024-01-22
8,1009,P-05,Hogar,12.0,118.65,Ana García,Norte,2024-01-23
9,1010,P-01,Electrónica,9.0,20.50,Luis Pérez,Norte,2024-01-24


### 3.4 Renombrar Columnas (`df.rename()`)

Es muy común que los nombres de las columnas contengan espacios extra, caracteres especiales o una capitalización inconsistente, lo que puede dificultar su uso. `df.rename()` nos permite cambiar estos nombres para estandarizarlos y hacerlos más amigables.

In [33]:
# Ejemplo: Renombrar algunas columnas para mayor consistencia
# Supongamos que queremos cambiar 'Id_venta' a 'ID_Venta' y 'Fecha_venta' a 'FechaVenta'
df_limpio = df_limpio.rename(columns={
    'Id_venta': 'ID_Venta',
    'Id_producto': 'ID_Producto',
    'CantidadVendida': 'Cantidad_Vendida',
    'PrecioUnitario': 'Precio_Unitario',
    'Fecha_venta': 'Fecha_Venta'
})

print("Nombres de columnas después de renombrar:")
print(df_limpio.columns)

Nombres de columnas después de renombrar:
Index(['ID_Venta', 'ID_Producto', 'Categoria', 'Cantidad_Vendida',
       'Precio_Unitario', 'Vendedor', 'Region', 'Fecha_Venta'],
      dtype='object')


---
## Paso 4: Corrección de Tipos de Datos

### 4.1 Fechas
Es común recibir fechas como texto con formatos distintos. Pandas puede inferir el formato automáticamente con `pd.to_datetime()`.


In [ ]:
print("Antes - tipo de dato:", df_limpio["Fecha_Venta"].dtype)
print("Valores únicos:", df_limpio["Fecha_Venta"].unique())


Antes - tipo de dato: object
Valores únicos: ['2024-01-15' '2024-01-16' '15/01/2024' '2024-01-18' '2024-01-19'
 '2024-01-20' '2024-01-21' '2024-01-22' '2024-01-23' '2024-01-24'
 '2024-01-25']


In [35]:
# Convertir a datetime (dayfirst=True maneja el formato DD/MM/YYYY y format='mixed' permite distintos formatos)
df_limpio["Fecha_Venta"] = pd.to_datetime(df_limpio["Fecha_Venta"], dayfirst=True, format='mixed')

print("Después - tipo de dato:", df_limpio["Fecha_Venta"].dtype)
print(df_limpio["Fecha_Venta"].head())

Después - tipo de dato: datetime64[ns]
0   2024-01-15
1   2024-01-16
2   2024-01-15
3   2024-01-18
4   2024-01-19
Name: Fecha_Venta, dtype: datetime64[ns]


### 4.2 Tipos numéricos


In [36]:
# Cantidad_vendida debe ser entero
df_limpio["Cantidad_Vendida"] = pd.to_numeric(
    df_limpio["Cantidad_Vendida"], errors="coerce"
).astype("Int64")

print(df_limpio[["Cantidad_Vendida", "Precio_Unitario"]].dtypes)


Cantidad_Vendida      Int64
Precio_Unitario     float64
dtype: object


---
## Paso 5: Identificación de Valores Sospechosos

Durante la exploración de datos es importante detectar valores que parecen fuera de lugar. A estos valores se los llama **outliers** (valores atípicos): registros que se alejan mucho del resto y que podrían ser errores de carga, errores de sistema, o casos genuinamente excepcionales.

Por ahora no vamos a usar herramientas estadísticas formales para detectarlos — eso lo haremos en el **Módulo 2** cuando veamos percentiles y distribuciones. Por ahora alcanza con usar `describe()` para inspeccionar los rangos y notar valores que llamen la atención.

> **¿Qué hacemos cuando encontramos uno?** Depende del contexto. Las opciones más comunes son: eliminarlo, reemplazarlo por la media de la columna, o dejarlo si tiene sentido de negocio. En este caso lo reemplazaremos por la media.


In [37]:
# Usamos describe() para inspeccionar los rangos de las columnas numéricas
df_limpio[["Cantidad_Vendida", "Precio_Unitario"]].describe()


,Cantidad_Vendida,Precio_Unitario
count,12.0,12.000000
mean,13.0,118.650000
std,7.410067,279.839081
min,5.0,15.000000
25%,8.75,19.875000
50%,11.0,20.500000
75%,13.5,55.912500
max,30.0,999.000000


In [ ]:
media_precio = df_limpio['Precio_Unitario'].mean()
mediana_precio = df_limpio['Precio_Unitario'].median()

print(f"Media del PrecioUnitario: {media_precio:.2f}")
print(f"Mediana (50%) del PrecioUnitario: {mediana_precio:.2f}")

print("\nEstadísticas descriptivas completas para PrecioUnitario:")
df_limpio['Precio_Unitario'].describe()

### Análisis de Media vs. Mediana para `PrecioUnitario`

Es importante entender la diferencia entre la **media (promedio)** y la **mediana (50%)**:

*   **Media:** Es la suma de todos los valores dividida por el número total de valores. Es sensible a los valores extremos (outliers).
*   **Mediana:** Es el valor central en un conjunto de datos ordenado. El 50% de los datos están por debajo de este valor y el 50% por encima. Es robusta ante los outliers.

**¿Qué nos dice la comparación?**

Después de la limpieza de datos y la eliminación del valor atípico (999.0):

*   Si la **media es significativamente mayor que la mediana**, como vemos en nuestro caso (`{media_precio:.2f}` vs `{mediana_precio:.2f}`), esto sugiere que hay **valores más altos** que están "tirando" el promedio hacia arriba. Es decir, aunque la mayoría de los productos se venden a precios más bajos o cercanos a la mediana, existen algunos productos con precios considerablemente más altos que influyen en la media.
*   Si la media y la mediana fueran muy similares, indicaría una distribución más simétrica de los precios.

Este análisis nos ayuda a comprender mejor la distribución de los precios y a identificar si hay una asimetría debido a la presencia de productos de alto valor, incluso después de haber manejado los errores obvios.

In [38]:
# El valor 999.0 en Precio_unitario llama la atención: es muy alto comparado con el resto.
# Calculamos la mediana excluyendo el valor sospechoso
mediana_sin_sospechoso = df_limpio.loc[df_limpio['Precio_Unitario'] != 999.0, 'Precio_Unitario'].median()

df_limpio.loc[df_limpio['Precio_Unitario'] == 999.0, 'Precio_Unitario'] = mediana_sin_sospechoso

print(f"Valor 999.0 reemplazado por la mediana: {mediana_sin_sospechoso:.2f}")
print(f"\nRango actual de 'Precio_Unitario':")
print(df_limpio['Precio_Unitario'].describe())


Valor 999.0 reemplazado por la mediana: 20.50

Rango actual de 'Precio_Unitario':
count     12.000000
mean      37.108333
std       38.420880
min       15.000000
25%       19.875000
50%       20.500000
75%       25.625000
max      118.650000
Name: Precio_Unitario, dtype: float64


---
## Paso 6: Eliminación de Duplicados

Pandas permite buscar duplicados sobre todas las columnas o solo sobre un subconjunto relevante.


In [39]:
# Verificar duplicados exactos (todas las columnas)
print(f"Duplicados exactos: {df_limpio.duplicated().sum()}")

# Verificar duplicados por Id_venta
print(f"Duplicados en Id_venta: {df_limpio.duplicated(subset='ID_Venta').sum()}")
print("\nRegistros duplicados:")
print(df_limpio[df_limpio.duplicated(subset='ID_Venta', keep=False)])


Duplicados exactos: 1
Duplicados en Id_venta: 1

Registros duplicados:
    ID_Venta ID_Producto    Categoria  Cantidad_Vendida  Precio_Unitario  \
9       1010        P-01  Electrónica                 9             20.5   
10      1010        P-01  Electrónica                 9             20.5   

      Vendedor Region Fecha_Venta  
9   Luis Pérez  Norte  2024-01-24  
10  Luis Pérez  Norte  2024-01-24  


In [40]:
# Eliminar duplicados exactos en todas las columnas
df_limpio = df_limpio.drop_duplicates(keep="first")
print(f"Filas después de eliminar duplicados exactos: {len(df_limpio)}")

# Resetear el índice después de eliminar filas para asegurar un índice continuo
df_limpio.reset_index(drop=True, inplace=True)
print("Índice del DataFrame reseteado.")

Filas después de eliminar duplicados exactos: 11
Índice del DataFrame reseteado.


---
## Paso 7: Validación Final

Siempre conviene hacer una revisión final para confirmar que el dataset quedó en buenas condiciones.


In [41]:
print("=== RESULTADOS ===")
print(f"Filas originales : {len(df)}")
print(f"Filas limpias    : {len(df_limpio)}")
print(f"\nNulos restantes  : {df_limpio.isnull().sum().sum()}")
print(f"Duplicados rest. : {df_limpio.duplicated().sum()}")
print(f"\nTipos de datos:")
print(df_limpio.dtypes)


=== RESULTADOS ===
Filas originales : 13
Filas limpias    : 11

Nulos restantes  : 0
Duplicados rest. : 0

Tipos de datos:
ID_Venta                     int64
ID_Producto                 object
Categoria                   object
Cantidad_Vendida             Int64
Precio_Unitario            float64
Vendedor                    object
Region                      object
Fecha_Venta         datetime64[ns]
dtype: object


## Aserciones
Si una condición que se espera que sea verdadera no se cumple, assert lanza un error (AssertionError), lo que detiene la ejecución y alerta inmediatamente sobre un problema. Esto es ideal para depuración y para asegurar que los datos cumplen con los requisitos mínimos antes de proceder con operaciones que podrían fallar o producir resultados incorrectos, como por ejemplo, verificar que no haya nulos ni duplicados antes de un análisis final.

In [42]:
assert not df_limpio.isnull().any().any(), "Se encontraron valores nulos en el DataFrame"

In [43]:
assert not df_limpio.duplicated().any(), "Existen filas duplicadas exactas en el dataset"

---
## Paso 8: Exportamos el dataset limpio  a un archivo CSV para su posterior análisis.

In [44]:
df_limpio.to_csv("ventas_limpio.csv", index=False)
print("DataFrame exportado a 'ventas_limpio.csv'")

DataFrame exportado a 'ventas_limpio.csv'


---
## Técnicas aplicadas

| Paso | Problema detectado | Técnica | Herramienta |
|---|---|---|---|
| Exploración | Visión general del dataset | `info()`, `describe()`, `value_counts()` | Pandas |
| Fila con Nulos | Fila `"Id_venta": 1012` con >60% nulos | Eliminar Fila | `dropna()` |
| Columna con Nulos | Columna `Notas_devolucion` con >80% nulos | Eliminar columna | `drop()` |
| Nulos | Nulos en categóricas y numéricas | Imputación con moda, media o mediana | `fillna()` |
| Strings | Mayúsculas/minúsculas inconsistentes y espacios extra | Normalización de texto | `str.strip()`, `str.title()`, `str.capitalize()` |
| Fechas | Dos formatos distintos (`YYYY-MM-DD` y `DD/MM/YYYY`) | Conversión flexible | `pd.to_datetime(..., format='mixed')` |
| Tipos | `Cantidad_vendida` almacenada como float | Casteo explícito | `astype(int)` |
| Valores sospechosos | `Precio_unitario = 999.0` visualmente anómalo | Reemplazo por mediana | Máscara booleana |
| Duplicados | Venta `Id_venta = 1010` duplicada | Eliminación por clave de negocio | `drop_duplicates()` |
